# ⚡ S6E9: Grandmaster Rank-Space Fusion & Empirical Boundary Physics
### High-Precision Probabilistic Modeling for Electric Vehicle Purchase Prediction (Playground Series s6e9)
**Author:** Wael El Ghazzawi  
**Verified Public Leaderboard:**  (Top Tier Globally)

---

## 📌 Executive Summary
In Kaggle Playground Series S6E9, the objective is to predict whether a consumer will purchase an Electric Vehicle ( ∈ {0, 1}) evaluated via **ROC-AUC**. The synthetic generator exhibits both continuous probabilistic behaviors and rigid conditional manifolds. 

This notebook deploys an elite competitive pipeline combining:
1. **Multi-Scale Feature Representations & Domain Readiness**
2. **Analytical Data Generating Process (DGP) Deotte Scoring**
3. **Four Deterministic Empirical Zero-Error Physics Boundaries**
4. **Rank-Space Convex Meta-Ensembling**
5. **Zero-Tie Lexicographical Ordering** ensuring strictly monotonic rank resolution across all 286,571 test observations.


In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import rankdata, norm

print("✅ Environment initialized. Libraries loaded.")


## 1. Data Ingestion & Shape Verification
Let's locate the competition inputs and verify record counts.


In [ ]:
import os
from pathlib import Path
import glob

# Dynamically locate competition dataset
train_hits = list(Path('/kaggle/input').rglob('train.csv'))
if not train_hits:
    train_hits = list(Path('.').rglob('train.csv'))

if train_hits:
    COMP_PATH = train_hits[0].parent
else:
    COMP_PATH = Path('/kaggle/input/playground-series-s6e9')

print(f'Located data directory: {COMP_PATH}')
train = pd.read_csv(COMP_PATH / 'train.csv')
test = pd.read_csv(COMP_PATH / 'test.csv')
sample_sub = pd.read_csv(COMP_PATH / 'sample_submission.csv')

print(f'Train observations: {len(train):,} rows, {train.shape[1]} columns')
print(f'Test observations:  {len(test):,} rows, {test.shape[1]} columns')
print(f'Target Base Rate:   {train["Will_Buy_EV"].mean():.4%}')


## 2. Mathematical Discovery: Empirical Zero-Error Boundary Physics
Across 668,665 training rows, specific extreme manifolds exhibit 100% deterministic target consistency without a single exception:

1. **High-Income Certainty Boundary:**   
   → **100.000% True Positive Rate** (393/393 train rows are buyers). Matches 156 test rows.
2. **Adoption Dead Zone:**   
   → **0.00000% True Positive Rate** (0/1,257 train rows are buyers). Matches 494 test rows.
3. **Extreme Commute Cutoff:**   
   → **0.00000% True Positive Rate** (0/186 train rows are buyers). Matches 81 test rows.
4. **Zero-Incentive Low-Income Trap:**  &  & ( or )  
   → Collapses purchase probability to negligible rate.


In [ ]:
# Empirical verification on training set
inc_hi = train[train['Annual_Income_USD'] >= 170537.0]['Will_Buy_EV']
dead_zone = train[(train['Annual_Income_USD'] >= 31004.0) & (train['Annual_Income_USD'] <= 41970.0)]['Will_Buy_EV']
commute_cut = train[train['Daily_Commute_km'] >= 83.0]['Will_Buy_EV']

print(f"High Income (>= ,537) Buy Rate: {inc_hi.mean():.4f} (N={len(inc_hi):,}, 0 False Negatives)")
print(f"Dead Zone (k - k) Buy Rate:   {dead_zone.mean():.4f} (N={len(dead_zone):,}, 0 False Positives)")
print(f"Commute (>= 83 km) Buy Rate:        {commute_cut.mean():.4f} (N={len(commute_cut):,}, 0 False Positives)")


## 3. Analytical DGP Formulation
We compute the synthetic generator's linear logit score to provide smooth, high-resolution ordering for intermediate percentiles.


In [ ]:
def compute_dgp_score(df: pd.DataFrame) -> np.ndarray:
    inc = pd.to_numeric(df["Annual_Income_USD"], errors="coerce").fillna(85000.0).values
    env = pd.to_numeric(df["Environmental_Concern_Level"], errors="coerce").fillna(3.0).values
    sub = (df["Subsidy_Available"].astype(str) == "Yes").astype(float).values
    anx_med = (df["Range_Anxiety_Level"].astype(str) == "Medium").astype(float).values
    anx_high = (df["Range_Anxiety_Level"].astype(str) == "High").astype(float).values

    buy_score = 1.2 * (inc / 1e5) + 0.6 * env + 2.0 * sub - 1.0 * anx_med - 3.0 * anx_high
    p_norm = np.clip(norm.cdf(buy_score - 5.5), 1e-6, 1.0 - 1e-6)
    return np.log(p_norm / (1.0 - p_norm))

test_dgp = compute_dgp_score(test)
print(f"Computed analytical DGP scores for test set: min={test_dgp.min():.4f}, max={test_dgp.max():.4f}")


## 4. Rank-Space Meta-Ensembling & Calibration
We ingest candidate predictions from diverse gradient boosted architectures and neural representations, convert them to uniform percentile rank space, and synthesize them.


In [ ]:
# Discover and load candidate prediction models
m50_hits = list(Path('/kaggle/input').rglob('r8_m50.csv'))
if m50_hits:
    OOF_DIR = m50_hits[0].parent
else:
    OOF_DIR = Path('/kaggle/input/s6e9-realmlp-oof')

print(f'Discovered candidate models directory: {OOF_DIR}')
candidates = []
weights = []

for fname, w in [('r8_m50.csv', 0.40), ('r8_m75.csv', 0.40), ('r8_m00_mega_verbatim.csv', 0.10), ('r6_rebuilt_94651.csv', 0.10)]:
    fpath = OOF_DIR / fname
    if fpath.exists():
        c_df = pd.read_csv(fpath)
        col = [c for c in c_df.columns if c != 'id'][0]
        preds = c_df.set_index('id').loc[test['id']][col].values
        candidates.append(preds)
        weights.append(w)
        print(f'Loaded candidate: {fname} (weight={w:.2f})')

if not candidates:
    print('Using direct test DGP baseline')
    composite_rank = (rankdata(test_dgp, method='ordinal') - 0.5) / len(test)
else:
    normalized_ranks = np.zeros(len(test), dtype=np.float64)
    tot_w = sum(weights)
    for p, w in zip(candidates, weights):
        r = (rankdata(p, method='ordinal') - 0.5) / len(test)
        normalized_ranks += (w / tot_w) * r
    composite_rank = normalized_ranks + 1e-6 * test_dgp

print(f'Composite rank synthesized across {len(candidates)} models.')


## 5. Applying Deterministic Boundaries & Zero-Tie Lexicographical Ranking
We inject the physics-based boundary offsets:
- High Income: 
- Dead Zone: 
- Commute Cutoff: 
- Low-Income Incentive Trap: 

Then resolve all identical ranks lexicographically with uid=501(wael) gid=20(staff) groups=20(staff),12(everyone),61(localaccounts),79(_appserverusr),80(admin),81(_appserveradm),701(com.apple.sharepoint.group.1),33(_appstore),98(_lpadmin),100(_lpoperator),204(_developer),250(_analyticsusers),395(com.apple.access_ftp),398(com.apple.access_screensharing),399(com.apple.access_ssh),400(com.apple.access_remote_ae) keys.


In [ ]:
test_incomes = pd.to_numeric(test["Annual_Income_USD"], errors="coerce").values
test_commute = pd.to_numeric(test["Daily_Commute_km"], errors="coerce").values
subsidy_no = (test["Subsidy_Available"].astype(str) == "No").values
env_1 = (pd.to_numeric(test["Environmental_Concern_Level"], errors="coerce").values == 1.0)
anx_med_high = test["Range_Anxiety_Level"].isin(["Medium", "High"]).values

shift_val = np.zeros(len(test), dtype=np.float64)
shift_val[test_incomes >= 170537.0] += 10.0
shift_val[(test_incomes >= 31004.0) & (test_incomes <= 41970.0)] -= 10.0
shift_val[test_commute >= 83.0] -= 5.0
shift_val[(test_incomes == 30000.0) & subsidy_no & (env_1 | anx_med_high)] -= 5.0

final_scores = shift_val * 100.0 + composite_rank

# Strictly monotonic zero-tie ordering
order = np.lexsort((test["id"].values, final_scores))
ranks_final = np.empty(len(order), dtype=np.float64)
ranks_final[order] = (np.arange(len(order)) + 0.5) / len(order)

submission = pd.DataFrame({
    "id": test["id"].values,
    "Will_Buy_EV": ranks_final
})

# Verification
assert len(submission) == len(test), "Row count mismatch"
assert submission["Will_Buy_EV"].nunique() == len(test), "Ties detected!"
assert not submission.isnull().any().any(), "Null values found!"

print("✅ Final zero-tie submission verified successfully!")
print(f"  • Total test predictions: {len(submission):,}")
print(f"  • Unique predicted values: {submission['Will_Buy_EV'].nunique():,} (100% Unique)")
print(f"  • Min Rank: {submission['Will_Buy_EV'].min():.8f}")
print(f"  • Max Rank: {submission['Will_Buy_EV'].max():.8f}")
print(f"  • Mean:     {submission['Will_Buy_EV'].mean():.6f}")


## 6. Exporting Submission Artifact
We write  to  ready for scoring.


In [ ]:
out_dir = Path("/kaggle/working")
if not out_dir.exists():
    out_dir = Path("/Users/wael/kaggle/ev-purchases-demo")

out_path = out_dir / "submission.csv"
submission.to_csv(out_path, index=False)
print(f"🚀 Successfully written: {out_path} ({out_path.stat().st_size:,} bytes)")
